In [ ]:
# Setup
from pathlib import Path
import sys, os
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from model import build_or_load_store, retrieve
from data import get_llm_answer
from utils import save_fig, plot_retrieval_scores

ARTEFACTS_DIR = PROJECT_ROOT / 'artefacts'
ARTEFACTS_DIR.mkdir(exist_ok=True)

In [ ]:
# 1. Build (or load) the vector store
store = build_or_load_store(force_rebuild=False)
print(f"Store contains {len(store)} chunks")

In [ ]:
# 2. Test retrieval on sample queries
queries = [
    "Who is the author of this project?",
    "What embedding model is used?",
    "What TCD modules does this project map to?",
]
for q in queries:
    hits = retrieve(q, store, top_k=3)
    print(f"\nQuery: {q}")
    for i, h in enumerate(hits):
        print(f"  [{i+1}] score={h['score']:.3f}  source={h['source']}")
        print(f"      {h['chunk'][:120]}...")

In [ ]:
# 3. Plot retrieval scores for the first query
q = queries[0]
hits = retrieve(q, store, top_k=5)
plot_retrieval_scores([h['score'] for h in hits])
save_fig('02_retrieval_scores.png')
plt.show()

In [ ]:
# 4. Generate an LLM answer with citations
chunks = [h['chunk'] for h in hits]
print(f"Backend auto-detected: OpenAI if OPENAI_API_KEY set, else local Ollama")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"\nQuery: {q}\n")

result = get_llm_answer(q, chunks, backend='auto')
print("=== ANSWER ===")
print(result['answer'])
print(f"\nBackend used: {result['backend']}")
print(f"Sources: {len(result['sources'])}")

In [ ]:
# 5. Save results for the evaluation notebook
import json
results = {
    'queries_tested': queries,
    'last_query': q,
    'backend_used': result['backend'],
    'top_k_chunks_count': len(chunks),
    'answer_chars': len(result['answer']),
    'corpus_chunks': len(store),
}
out = ARTEFACTS_DIR / 'rag_results.json'
with open(out, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f"Saved → {out}")